# 03 — EDA: Weather & Temporal Patterns

## Purpose
This notebook studies **when and under what conditions** people buy different products.
Understanding these patterns is critical because temperature, day of week, and time of day
are all features our forecasting model will use.

We answer:
- Do cold days and warm days drive different product preferences?
- Which products are most popular on each day of the week?
- Does product preference change by morning, afternoon, or night?
- How do monthly sales trend over time per branch?
- What is the average ticket value, and does temperature affect it?

## Input
`data/intermediate/datanomodifier.csv`

## Run order
Run after `01_data_cleaning.ipynb`.

In [ ]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROCESSED_DIR    = PROCESSED_DIR
    WEATHER_DIR      = WEATHER_DIR
    RAW_DIR          = RAW_DIR
    INTERMEDIATE_DIR = INTERMEDIATE_DIR


## Setup — Load and clean data

## Analysis 1 — Top items on cold vs warm days (overall)

We split all sales into two groups:
- **warm**: days where `tavg >= 25°C` (Monterrey summer threshold)
- **cold**: days where `tavg < 25°C`

Then we rank the top 20 items in each group. Products that rank differently between
the two groups are the ones most sensitive to temperature — key signals for the model.

In [ ]:
N = 20

top_cold = (
    base[base["cold_or_warm"] == "cold"]
    .groupby("item", as_index=False)["quantity"].sum()
    .sort_values("quantity", ascending=False)
    .head(N).reset_index(drop=True)
)
top_cold.index += 1

top_warm = (
    base[base["cold_or_warm"] == "warm"]
    .groupby("item", as_index=False)["quantity"].sum()
    .sort_values("quantity", ascending=False)
    .head(N).reset_index(drop=True)
)
top_warm.index += 1

print("=== Top items — COLD days (tavg < 25°C) ===")
display(top_cold)
print("\n=== Top items — WARM days (tavg >= 25°C) ===")
display(top_warm)

## Analysis 2 — Top items cold vs warm, per branch

The overall view above mixes all branches. Here we break it down per branch
because each location has a different customer demographic and product mix.
A product that tops warm-day sales at Hotel Kavia might not even appear at Credi Club.

In [ ]:
TOP_N = 10

for branch in sorted(base["sucursal"].dropna().unique()):
    print(f"\n{'='*55}\n  {branch}\n{'='*55}")
    for weather in ["cold", "warm"]:
        g = (
            base[(base["sucursal"] == branch) & (base["cold_or_warm"] == weather)]
            .groupby("item", as_index=False)["quantity"].sum()
            .rename(columns={"quantity": "qty_sold"})
            .sort_values("qty_sold", ascending=False)
            .head(TOP_N).reset_index(drop=True)
        )
        if len(g):
            g.index += 1
            print(f"\n  --- {weather.upper()} days ---")
            display(g)

## Analysis 3 — Top items by day of week

Weekdays and weekends have different traffic patterns. A croissant might be a
top seller on Monday mornings but not on Sundays.

We show the top 10 items per day (Monday → Sunday) sorted in calendar order.

In [ ]:
# Map day names to a sort order (Spanish)
day_order = {"lunes": 0, "martes": 1, "miércoles": 2, "miercoles": 2,
             "jueves": 3, "viernes": 4, "sábado": 5, "sabado": 5, "domingo": 6}

TOP_N = 10

top_by_day = (
    base.groupby(["day_name", "item"], as_index=False)["quantity"]
        .sum().rename(columns={"quantity": "units_sold"})
)
top_by_day["day_rank"] = top_by_day["day_name"].str.lower().map(day_order)
top_by_day = (
    top_by_day.sort_values(["day_rank", "units_sold"], ascending=[True, False])
              .groupby("day_name", as_index=False).head(TOP_N)
)

ordered_days = top_by_day[["day_name","day_rank"]].drop_duplicates().sort_values("day_rank")["day_name"].tolist()

for day in ordered_days:
    print(f"\n=== {day.upper()} ===")
    t = top_by_day[top_by_day["day_name"] == day][["item","units_sold"]].reset_index(drop=True)
    t.index += 1
    display(t)

## Analysis 4 — Top items by time of day (overall)

**Day parts:**
- `morning` = 6:00 – 11:59 (breakfast rush)
- `afternoon` = 12:00 – 18:59 (lunch + afternoon snack)
- `night` = 19:00 – 5:59 (dinner, late)

Bakery and café items behave very differently by day part. Croissants peak in the morning;
lunches peak at noon. This is a critical feature for stock planning.

In [ ]:
TOP_N = 10

top_by_part = (
    base.groupby(["day_part", "item"], as_index=False)["quantity"]
        .sum().rename(columns={"quantity": "units_sold"})
        .sort_values(["day_part", "units_sold"], ascending=[True, False])
        .groupby("day_part", as_index=False).head(TOP_N)
)

for part in ["morning", "afternoon", "night"]:
    t = top_by_part[top_by_part["day_part"] == part][["item","units_sold"]].reset_index(drop=True)
    if len(t):
        t.index += 1
        print(f"\n=== {part.upper()} ===")
        display(t)

## Analysis 5 — Top items by day part, per branch

Same as above but broken down by branch. This is the most granular behavioral view
before we move to feature engineering.

In [ ]:
for branch in sorted(base["sucursal"].dropna().unique()):
    print(f"\n{'='*55}\n  {branch}\n{'='*55}")
    for part in ["morning", "afternoon", "night"]:
        t = (
            base[(base["sucursal"] == branch) & (base["day_part"] == part)]
            .groupby("item", as_index=False)["quantity"].sum()
            .rename(columns={"quantity": "units_sold"})
            .sort_values("units_sold", ascending=False)
            .head(10).reset_index(drop=True)
        )
        if len(t):
            t.index += 1
            print(f"\n  --- {part.upper()} ---")
            display(t)

## Analysis 6 — Monthly sales bar charts per branch

This chart shows how total revenue evolved month by month for each branch.
Look for:
- Seasonal peaks (e.g., December holiday season)
- Growth or decline trends
- Missing months (branch may have been closed)
- Dips that may correspond to holidays or external events

Each bar represents one month's total ticket sales in MXN.

In [ ]:
import math

df_monthly = df[(df["is_modifier"] == False) & (df["action"] == "Venta")].copy()
df_monthly["month"] = df_monthly["operating_date"].dt.to_period("M").dt.to_timestamp()

monthly = (
    df_monthly.dropna(subset=["operating_date", "sucursal"])
              .groupby(["sucursal", "month"], as_index=False)["total_ticket"]
              .sum().rename(columns={"total_ticket": "sales"})
)

stores = sorted(monthly["sucursal"].unique())
ncols = 3
nrows = math.ceil(len(stores) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = axes.flatten()

for i, branch in enumerate(stores):
    g = monthly[monthly["sucursal"] == branch].sort_values("month")
    axes[i].bar(g["month"].dt.strftime("%Y-%m"), g["sales"], color="#169830")
    axes[i].set_title(branch, fontsize=9)
    axes[i].tick_params(axis="x", rotation=90, labelsize=7)
    axes[i].yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    axes[i].grid(axis="y", alpha=0.2)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Monthly Sales Revenue (MXN) per Branch", fontsize=13)
plt.tight_layout()
plt.show()

## Analysis 7 — Average ticket value by year

The average ticket value tells us how much a customer spends per visit on average.
Tracking this year over year reveals whether the business is moving upmarket
(customers spending more per visit) or if volume growth is coming from more customers.

In [ ]:
df_ticket = df[(df["is_modifier"] == False) & (df["action"] == "Venta")].copy()
df_ticket["year"] = df_ticket["operating_date"].dt.year

# Overall average ticket per year
mean_by_year = (
    df_ticket.dropna(subset=["operating_date", "total_ticket"])
             .groupby("year", as_index=False)["total_ticket"]
             .mean().rename(columns={"total_ticket": "avg_ticket_mxn"})
)

print("=== Average ticket value by year (all branches) ===")
print(mean_by_year.to_string(index=False))

# Average ticket per year per branch
mean_by_year_branch = (
    df_ticket.dropna(subset=["operating_date", "total_ticket", "sucursal"])
             .groupby(["sucursal", "year"], as_index=False)["total_ticket"]
             .mean().rename(columns={"total_ticket": "avg_ticket_mxn"})
             .sort_values(["sucursal", "year"])
)
print("\n=== Average ticket value by year per branch ===")
display(mean_by_year_branch)

## Analysis 8 — Cold vs Warm ticket box plot (overall)

Does temperature affect how much customers spend per visit?

We build a daily summary (one row per day) and compare:
- Number of daily transactions on cold days vs warm days
- We use a **bootstrap confidence interval** to test whether the difference is statistically real
  or just random variation

**Bootstrap** = resample each group thousands of times and measure the distribution of differences.
If the 95% confidence interval doesn't include zero, the difference is statistically significant.

In [ ]:
# Build daily ticket count (one row per day)
daily = (
    df[(df["is_modifier"] == False) & (df["action"] == "Venta")]
    .drop_duplicates("pdv_txn_id")
    .groupby("operating_date", as_index=False)
    .agg(tavg=("tavg", "mean"), ticket_count=("pdv_txn_id", "count"))
)
daily["temp_group"] = np.where(daily["tavg"] >= 25, "warm", "cold")

cold = daily.loc[daily["temp_group"] == "cold", "ticket_count"].values
warm = daily.loc[daily["temp_group"] == "warm", "ticket_count"].values

# Box plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([cold, warm], tick_labels=["Cold (<25°C)", "Warm (≥25°C)"], showmeans=True)
ax.set_title("Daily Transaction Count: Cold vs Warm (All Branches)")
ax.set_ylabel("Transactions per day")
ax.grid(axis="y", alpha=0.2)
plt.show()

# Bootstrap confidence interval
n = min(len(cold), len(warm))
rng = np.random.default_rng(42)
diffs = np.array([rng.choice(warm, n, replace=True).mean() - rng.choice(cold, n, replace=True).mean() for _ in range(5000)])
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

print(f"Cold days: {len(cold)} | Warm days: {len(warm)}")
print(f"Mean diff (warm - cold): {diffs.mean():,.1f} transactions/day")
print(f"95% CI: [{ci_low:,.1f}, {ci_high:,.1f}]")
print("Significant" if ci_low > 0 or ci_high < 0 else "Not significant at 95% level")

## Summary

Key takeaways:
- Temperature creates distinct product demand patterns — `cold_or_warm` is a valuable feature
- Day of week shows consistent patterns — weekday vs weekend is predictable
- Day part (morning/afternoon/night) reveals very different product mixes — useful for stock timing
- Monthly charts show seasonal trends and any outlier months

**Next step:** `04_eda_hourly_patterns.ipynb` — deep dive into hour-by-hour and holiday patterns.